<a href="https://colab.research.google.com/github/pauladevcich/pauladevcich.github.io/blob/main/LAB1_steps4%265.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**STEP 4 - GRAPHICAL REPRESENTATION**

In [ ]:
##Load all datasets

file_path = "/content/drive/MyDrive/PoliTo/2do cuatri/DECISION_AID/LAB1/Dataset_DAPPC_2026.xlsx"

df_3a = pd.read_excel(file_path, sheet_name="Dataset_step_3a")
df_3b = pd.read_excel(file_path, sheet_name="Dataset_step_3b")
df_3c = pd.read_excel(file_path, sheet_name="Dataset_step_3c")
df_3d = pd.read_excel(file_path, sheet_name="Dataset_step_3d")

datasets = {
    "3a": df_3a,
    "3b": df_3b,
    "3c": df_3c,
    "3d": df_3d
}

for name, df in datasets.items():
    df.columns = df.columns.str.strip()

print("Loaded datasets:", list(datasets.keys()))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PoliTo/2do cuatri/DECISION_AID/LAB1/Dataset_DAPPC_2026.xlsx'

In [ ]:
##Define variable groups

continuous_vars = [
    "height","weight",
    "first_pCO2","std_pCO2","first_pH","std_pH","first_pO2","std_pO2",
    "first_lactate","std_lactate","first_PT","std_PT","first_PTT","std_PTT",
    "first_bilirubin","std_bilirubin","first_chloride","std_chloride",
    "first_creatinine","std_creatinine","first_glycemia","std_glycemia",
    "first_hemoglobin","std_hemoglobin","first_phosphate","std_phosphate",
    "first_platelets","std_platelets","first_potassium","std_potassium",
    "first_sodium","std_sodium","first_wbc","std_wbc",
    "first_diastolic_blood_pressure","std_diastolic_blood_pressure",
    "first_mean_blood_pressure","std_mean_blood_pressure",
    "first_systolic_blood_pressure","std_systolic_blood_pressure",
    "first_heart_rate","std_heart_rate","first_respiratory_rate","std_respiratory_rate",
    "first_temperature","std_temperature","first_PEEP","std_PEEP",
    "first_PIP","std_PIP","first_mean_airway_pressure","std_mean_airway_pressure",
    "first_plateau_pressure","std_plateau_pressure","vent_duration_hours"
]

integer_vars = [
    "age","sedative_duration_hours","comorb_total"
]

categorical_vars = [
    "gender","charlson_comorbidity_index","gcs","sofa","sirs","sapsii",
    "vasopressors","sedatives","neuromuscular_blockers","steroids","opioids",
    "cardiac_cardiovascular","respiratory_pulmonary","metabolic_endocrine_renal",
    "neurological_neuromuscular_psychiatric","systemic_immune_oncologic"
]

# Keep only columns that actually exist in ALL / at least current datasets
all_cols = set().union(*[set(df.columns) for df in datasets.values()])

continuous_vars = [c for c in continuous_vars if c in all_cols]
integer_vars = [c for c in integer_vars if c in all_cols]
categorical_vars = [c for c in categorical_vars if c in all_cols]

print("Continuous:", len(continuous_vars))
print("Integer:", len(integer_vars))
print("Categorical/Binary:", len(categorical_vars))

In [ ]:
##Helper function for continuous variables

def plot_continuous_by_variable(datasets, variables, outcome_col="outcome"):

    n_rows = len(variables)
    n_cols = len(datasets)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4*n_cols, 3*n_rows),
        squeeze=False
    )

    dataset_names = list(datasets.keys())

    for i, var in enumerate(variables):
        for j, ds_name in enumerate(dataset_names):

            ax = axes[i, j]
            df = datasets[ds_name]

            if var not in df.columns:
                ax.axis("off")
                continue

            temp = df[[var, outcome_col]].dropna()

            if temp.empty:
                ax.text(0.5, 0.5, "No data", ha="center", va="center")
                ax.set_axis_off()
                continue

            sns.boxplot(data=temp, x=outcome_col, y=var, ax=ax)

            # Titles
            if i == 0:
                ax.set_title(ds_name)

            # Row labels
            if j == 0:
                ax.set_ylabel(var)
            else:
                ax.set_ylabel("")

            ax.set_xlabel("")

    plt.suptitle("Continuous variables — comparison across datasets", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
##Helper function for integer variables

def plot_integer_by_variable(datasets, variables, outcome_col="outcome", unique_threshold=10):

    n_rows = len(variables)
    n_cols = len(datasets)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4*n_cols, 3*n_rows),
        squeeze=False
    )

    dataset_names = list(datasets.keys())

    for i, var in enumerate(variables):
        for j, ds_name in enumerate(dataset_names):

            ax = axes[i, j]
            df = datasets[ds_name]

            if var not in df.columns:
                ax.axis("off")
                continue

            temp = df[[var, outcome_col]].dropna()

            if temp.empty:
                ax.text(0.5, 0.5, "No data", ha="center", va="center")
                ax.set_axis_off()
                continue

            n_unique = temp[var].nunique()

            if n_unique <= unique_threshold:
                sns.countplot(data=temp, x=var, hue=outcome_col, ax=ax)
                if ax.get_legend() is not None and not (i == 0 and j == n_cols-1):
                    ax.get_legend().remove()
            else:
                sns.boxplot(data=temp, x=outcome_col, y=var, ax=ax)

            if i == 0:
                ax.set_title(ds_name)

            if j == 0:
                ax.set_ylabel(var)
            else:
                ax.set_ylabel("")

            ax.set_xlabel("")
            ax.tick_params(axis='x', rotation=45)

    plt.suptitle("Integer variables — comparison across datasets", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
##Helper function for categorical/binary variables

def plot_categorical_by_variable(datasets, variables, outcome_col="outcome"):

    n_rows = len(variables)
    n_cols = len(datasets)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4*n_cols, 3*n_rows),
        squeeze=False
    )

    dataset_names = list(datasets.keys())

    for i, var in enumerate(variables):
        for j, ds_name in enumerate(dataset_names):

            ax = axes[i, j]
            df = datasets[ds_name]

            if var not in df.columns:
                ax.axis("off")
                continue

            temp = df[[var, outcome_col]].dropna()

            if temp.empty:
                ax.text(0.5, 0.5, "No data", ha="center", va="center")
                ax.set_axis_off()
                continue

            sns.countplot(data=temp, x=var, hue=outcome_col, ax=ax)

            if ax.get_legend() is not None and not (i == 0 and j == n_cols-1):
                ax.get_legend().remove()

            if i == 0:
                ax.set_title(ds_name)

            if j == 0:
                ax.set_ylabel(var)
            else:
                ax.set_ylabel("")

            ax.set_xlabel("")
            ax.tick_params(axis='x', rotation=45)

    plt.suptitle("Categorical/Binary variables — comparison across datasets", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
##Plot continuous variables

plot_continuous_by_variable(datasets, continuous_vars)

In [ ]:
##Plot integer variables

plot_integer_by_variable(datasets, integer_vars)

In [ ]:
##Plot categorical/binary variables

plot_categorical_by_variable(datasets, categorical_vars)

**STEP 5 - OUTLIER DETECTION**

In [ ]:
##Helper function: IQR outlier detection

def detect_outliers_iqr(df, variables):
    results = []

    for var in variables:
        if var not in df.columns:
            continue

        x = df[var].dropna()

        if len(x) == 0:
            results.append({
                "variable": var,
                "n_outliers": np.nan,
                "pct_outliers": np.nan,
                "lower_bound": np.nan,
                "upper_bound": np.nan
            })
            continue

        Q1 = x.quantile(0.25)
        Q3 = x.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        mask = (df[var] < lower_bound) | (df[var] > upper_bound)
        n_outliers = mask.sum()
        pct_outliers = 100 * n_outliers / df[var].notna().sum()

        results.append({
            "variable": var,
            "n_outliers": int(n_outliers),
            "pct_outliers": round(pct_outliers, 2),
            "lower_bound": round(lower_bound, 3),
            "upper_bound": round(upper_bound, 3)
        })

    return pd.DataFrame(results).sort_values("pct_outliers", ascending=False)

In [ ]:
##Run outlier detection for all datasets

outlier_results = {}

vars_for_outliers = continuous_vars + integer_vars

for name, df in datasets.items():
    outlier_results[name] = detect_outliers_iqr(df, vars_for_outliers)
    print(f"\n=== Dataset {name} ===")
    display(outlier_results[name])

In [ ]:
##Show only variables with at least 1 outlier

for name, table in outlier_results.items():
    print(f"\n=== Dataset {name} - variables with outliers ===")
    display(table[table["n_outliers"] > 0])

In [ ]:
##Compare total outliers across datasets

summary_outliers = []

for name, table in outlier_results.items():
    total_outliers = table["n_outliers"].sum()
    vars_with_outliers = (table["n_outliers"] > 0).sum()

    summary_outliers.append({
        "dataset": name,
        "total_outliers": int(total_outliers),
        "variables_with_outliers": int(vars_with_outliers)
    })

summary_outliers = pd.DataFrame(summary_outliers)
display(summary_outliers.sort_values("total_outliers"))

In [ ]:
##Outlier counts by class

def detect_outliers_iqr_by_class(df, variables, outcome_col="outcome"):
    results = []

    for outcome_value in sorted(df[outcome_col].dropna().unique()):
        subset = df[df[outcome_col] == outcome_value]

        for var in variables:
            if var not in subset.columns:
                continue

            x = subset[var].dropna()

            if len(x) == 0:
                continue

            Q1 = x.quantile(0.25)
            Q3 = x.quantile(0.75)
            IQR = Q3 - Q1

            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            mask = (subset[var] < lower_bound) | (subset[var] > upper_bound)
            n_outliers = mask.sum()
            pct_outliers = 100 * n_outliers / subset[var].notna().sum()

            results.append({
                "class": outcome_value,
                "variable": var,
                "n_outliers": int(n_outliers),
                "pct_outliers": round(pct_outliers, 2)
            })

    return pd.DataFrame(results)

In [ ]:
##Run outlier detection by class

outlier_results_by_class = {}

for name, df in datasets.items():
    outlier_results_by_class[name] = detect_outliers_iqr_by_class(df, vars_for_outliers)
    print(f"\n=== Dataset {name} - outliers by class ===")
    display(outlier_results_by_class[name])

In [ ]:
##Heatmap to compare datasets

comparison_tables = []

for name, table in outlier_results.items():
    temp = table[["variable", "pct_outliers"]].copy()
    temp = temp.rename(columns={"pct_outliers": name})
    comparison_tables.append(temp)

outlier_compare = comparison_tables[0]
for temp in comparison_tables[1:]:
    outlier_compare = outlier_compare.merge(temp, on="variable", how="outer")

outlier_compare = outlier_compare.set_index("variable")

plt.figure(figsize=(8, max(6, len(outlier_compare)*0.35)))
sns.heatmap(outlier_compare, annot=True, fmt=".2f", cmap="Reds")
plt.title("Percentage of outliers by variable and dataset")
plt.tight_layout()
plt.show()